# 지식 기반 의미역 분석 (Knowledge-based SRL)

### FrameNet
- 동사 의미를 “상황 단위(프레임)”로 보고, 프레임 요소(FEs)(Donor, Recipient, Theme …)를 제공

### PropBank
- 동사별 roleset(예: give.01)에 대해 Arg0/Arg1/Arg2…의 역할 설명을 제공

In [15]:
# ============================================
# 지식 기반 SRL 데모: FrameNet + PropBank
# ============================================

# (처음 1회만) 필요한 리소스 다운로드
import nltk
nltk.download('framenet_v17')   # FrameNet 데이터
nltk.download('propbank')       # PropBank 메타데이터(roleset 정의)
nltk.download('punkt')          # 토큰화 등 기본 리소스

[nltk_data] Downloading package framenet_v17 to /root/nltk_data...
[nltk_data]   Unzipping corpora/framenet_v17.zip.
[nltk_data] Downloading package propbank to /root/nltk_data...
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

### FrameNet은 동사를 중심으로 “상황 = 프레임”을 정의
- 각 프레임에는 Frame Elements(FEs)라는 역할(Role) 목록이 있음 (예: Giver, Recipient, Goods 등)
- 이 함수는 해당 동사와 연결된 프레임을 찾아 프레임 이름 + 역할 요소 목록을 출력하는 기능

In [28]:
from nltk.corpus import framenet as fn
from nltk.corpus import propbank as pb

# --------------------------------------------
# 1) FrameNet: 동사 → 관련 프레임 → 프레임 요소(FEs) 확인
# --------------------------------------------
def show_framenet_roles(verb: str, pick_first=True):
    """
    FrameNet에서 특정 동사(원형, lemma)와 연결된 '프레임(frame)'을 찾고,
    그 프레임 안에서 정의된 '역할 요소(Frame Elements)'를 보기 좋게 출력하는 함수.
    """

    print(f"\n[FrameNet] verb = '{verb}'")

    # FrameNet에서 해당 동사와 연결된 프레임 목록 가져오기
    frames = fn.frames_by_lemma(verb)
    if not frames:
        print(" - 관련 프레임을 찾지 못했습니다.")
        return None

    # 여러 프레임이 나올 수 있으므로, 기본적으로 첫 번째 것만 확인
    # (보통 가장 일반적이고 직관적인 프레임이 앞에 위치)
    frame = frames[0] if pick_first else frames

    # 만약 여러 프레임 리스트를 그대로 반환하는 경우
    if isinstance(frame, list):
        print(f" - 후보 프레임 {len(frame)}개: {[f.name for f in frame]}")
        return frame

    # 선택된 프레임 이름 출력
    print(f" - 선택 프레임: {frame.name}")

    # 프레임에 정의된 Frame Elements(FEs) 확인
    # 각 역할 요소 이름(fe_name)과 Core/Peripheral 구분을 가져옴
    fe_rows = []
    for fe_name, fe_obj in frame.FE.items():
        fe_rows.append((fe_name, fe_obj.coreType))  # coreType은 역할의 성격(Core/Peripheral)

    # 보기 좋게 출력
    print(" - Frame Elements (역할 요소):")
    for name, ctype in sorted(fe_rows):
        print(f"   · {name:<15} | type={ctype}")

    return frame

- roleset = PropBank에서 특정 동사의 의미/용법 단위 (예: give.01 = “주다”, give.02 = “포기하다”)

- 각 roleset은 Arg0, Arg1, Arg2 … 식으로 역할 정의를 제공

  - 보통 Arg0 = Agent, Arg1 = Theme, Arg2 = Recipient 등

- 이 함수는 동사에 연결된 roleset을 불러와 역할 설명을 출력하는 도구

In [29]:
# --------------------------------------------
# 2) PropBank: 동사 → roleset(예: give.01) → Arg0/Arg1 정의 확인
# --------------------------------------------
def show_propbank_roles(verb: str, limit=2):
    """
    PropBank에서 특정 동사(lemma) 기준으로 roleset 목록을 찾고,
    각 roleset에 정의된 Arg0, Arg1 ... 의미역 설명을 출력하는 함수.
    """

    print(f"\n[PropBank] verb = '{verb}'")
    try:
        # pb.rolesets(verb)는 Element 객체 리스트를 반환함
        # 단순 문자열이 아니므로 'id' 속성에서 roleset 이름을 추출해야 함
        # 예: ['give.01', 'give.02', ...]
        rolesets = [rs.get('id') for rs in pb.rolesets(verb) if rs.get('id')]
    except Exception as e:
        # 실행 환경에 PropBank 리소스가 없을 수 있음 → 예외 처리
        print(" - PropBank 조회가 어려운 환경입니다. (간단 요약만 표시)")
        print("   예) give.01: Arg0=Agent(주는 사람), Arg1=Theme(주는 것), Arg2=Recipient(받는 사람)")
        return

    # 만약 해당 동사에 roleset이 없는 경우
    if not rolesets:
        print(" - 관련 roleset이 없습니다.")
        return

    # 찾은 roleset 목록 출력 (limit 개수까지만)
    print(f" - roleset 후보: {rolesets[:limit]}{' ...' if len(rolesets) > limit else ''}")
    for rid in rolesets[:limit]:
        # 특정 roleset 객체 가져오기 (예: give.01)
        rs = pb.roleset(rid)

        # roleset 이름 출력
        print(f"   ▶ {rid}: {getattr(rs, 'name', '')}")

        # roleset 안에 정의된 Arg 목록 출력
        for role in getattr(rs, 'roles', []):
            argn = role['n']       # Arg 번호 (0, 1, 2, …)
            descr = role['descr']  # Arg의 의미 설명
            print(f"     - Arg{argn}: {descr}")

In [24]:
# --------------------------------------------
# 3) 예문에 역할 붙여보기(지식 기반 매핑 예시)
#    - 실제 SRL 러너가 아니라 '교육용 매핑'입니다.
#    - 프레임/roleset 정의를 사람이 읽고 붙이는 방식의 데모.
# --------------------------------------------
def demo_assign_roles(sentence: str):
    """
    간단 교육용 데모:
    'Mary gave John a book.' 예문에 대해
    - FrameNet 'Giving' 프레임 관점 역할명
    - PropBank 'give.01' 관점 Arg 매핑
    을 보기 좋게 요약 출력.
    """
    print("\n[예문] ", sentence)
    print("\n[FrameNet 관점: Giving 프레임 가정]")
    print(" - Giver      : Mary")
    print(" - Recipient  : John")
    print(" - Goods      : a book")

    print("\n[PropBank 관점: give.01 가정]")
    print(" - Arg0 (Agent)     : Mary")
    print(" - Arg1 (Theme)     : a book")
    print(" - Arg2 (Recipient) : John")

In [27]:
# --------------------------------------------
# 실행 데모
# --------------------------------------------
if __name__ == "__main__":
    # 1) FrameNet: 'give' 동사의 프레임/역할 요소 살펴보기
    frame = show_framenet_roles("give")

    # 2) PropBank: 'give' 동사의 roleset과 Arg 정의 살펴보기
    show_propbank_roles("give", limit=2)

    # 3) 예문에 역할 부착(지식 기반 매핑 예시)
    demo_assign_roles("Mary gave John a book.")


[FrameNet] verb = 'give'
 - 선택 프레임: Attempt_means
 - Frame Elements (역할 요소):
   · Agent           | type=Core
   · Circumstances   | type=Extra-Thematic
   · Degree          | type=Peripheral
   · Depictive       | type=Extra-Thematic
   · Domain          | type=Extra-Thematic
   · Duration        | type=Extra-Thematic
   · Frequency       | type=Extra-Thematic
   · Goal            | type=Core-Unexpressed
   · Manner          | type=Peripheral
   · Means           | type=Core
   · Outcome         | type=Extra-Thematic
   · Particular_iteration | type=Extra-Thematic
   · Place           | type=Peripheral
   · Purpose         | type=Peripheral
   · Time            | type=Peripheral

[PropBank] verb = 'give'
 - roleset 후보: ['give.01', 'give.02'] ...
   ▶ give.01: 
   ▶ give.02: 

[예문]  Mary gave John a book.

[FrameNet 관점: Giving 프레임 가정]
 - Giver      : Mary
 - Recipient  : John
 - Goods      : a book

[PropBank 관점: give.01 가정]
 - Arg0 (Agent)     : Mary
 - Arg1 (Theme)     : a book
 - A

- 추가 예제

   "Mary gave John a book."

   "John sent a letter to Mary."

   "He sold the car to Mary."


In [32]:
# --------------------------------------------
# 3) 예문에 역할 붙여보기 (아주 단순한 규칙 기반 데모)
#    - 목적: FrameNet/PropBank 역할 이름을 감각적으로 보여주기
#    - 제한: 고정 어순(SVO) & 간단 전치사 패턴만 처리 (정확도보다 이해도 중심)
# --------------------------------------------

def demo_assign_roles(sentence: str):
    """
    입력 문장에 대해, '주다/보내다/팔다' 유형 예문을
    교육용 규칙으로 역할을 붙여서 보기 좋게 출력.
    - FrameNet 관점: Giver/Sender/Seller, Recipient/Buyer, Goods 등
    - PropBank 관점: Arg0/Arg1/Arg2 등
    """
    s = sentence.strip()

    # ===== 1) 아주 단순한 토큰 나누기 =====
    # 예: "Mary gave John a book."
    tokens = s.replace(".", "").split()

    # ===== 2) 아주 단순한 패턴 인식 =====
    # (1) give/send류: "SUBJ gave/ sent OBJ1 to OBJ2" 또는 "SUBJ gave OBJ2 OBJ1"
    # (2) sell류:      "SUBJ sold OBJ1 to OBJ2"
    verb = None
    if any(w.lower() in {"gave", "give", "sent", "send"} for w in tokens):
        verb = "give/ send"
    elif any(w.lower() in {"sold", "sell"} for w in tokens):
        verb = "sell"

    # 기본 값
    subj = obj1 = obj2 = None

    # "to" 위치 찾기 (수취자/수혜자 표현 단서)
    to_idx = next((i for i, w in enumerate(tokens) if w.lower() == "to"), -1)

    # 매우 단순한 규칙: [SUBJ] [VERB] [X] [Y] (to [Z] 형태도 허용)
    # 예) Mary gave John a book.  → SUBJ=Mary, OBJ2=John, OBJ1=a book
    # 예) John sent a letter to Mary. → SUBJ=John, OBJ1=a letter, OBJ2=Mary
    try:
        # 주어는 문장 첫 단어(또는 첫 토큰)로 가정 (대문자 시작 이름 중심 데모)
        subj = tokens[0]

        # 동사 위치
        v_idx = next(i for i, w in enumerate(tokens)
                     if w.lower() in {"gave", "give", "sent", "send", "sold", "sell"})

        # "to"가 있으면: OBJ1 = 동사 다음~to 전까지, OBJ2 = to 다음
        if to_idx > v_idx:
            obj1 = " ".join(tokens[v_idx+1:to_idx]) or None
            obj2 = " ".join(tokens[to_idx+1:]) or None
        else:
            # "to"가 없으면: 동사 뒤에 두 덩어리라고 가정 (예: gave John a book)
            # 마지막 두 단어를 OBJ1, 나머지를 OBJ2로 나누는 식 등 다양 가능
            # 여기서는 간단히: 동사 다음 단어 = OBJ2, 나머지 = OBJ1 로 가정
            # (Mary gave John a book → OBJ2=John, OBJ1=a book)
            if v_idx + 2 < len(tokens):
                obj2 = tokens[v_idx+1]
                obj1 = " ".join(tokens[v_idx+2:])
            elif v_idx + 1 < len(tokens):
                obj1 = " ".join(tokens[v_idx+1:])
    except StopIteration:
        pass  # 동사를 못 찾으면 None 유지

    # ===== 3) 역할 이름 매핑 (교육용) =====
    # FrameNet 관점 이름
    if verb == "give/ send":
        fr_roles = {
            "Giver/Sender": subj,
            "Recipient":    obj2,
            "Goods":        obj1
        }
        pb_roles = {
            "Arg0 (Agent)":    subj,
            "Arg1 (Theme)":    obj1,
            "Arg2 (Recipient)":obj2
        }
        frame_name = "Giving / Sending"
        roleset_name = "give.01 / send.01 (가정)"
    elif verb == "sell":
        fr_roles = {
            "Seller":  subj,
            "Buyer":   obj2,
            "Goods":   obj1
        }
        pb_roles = {
            "Arg0 (Seller/Agent)": subj,
            "Arg1 (Thing sold)":   obj1,
            "Arg2 (Buyer)":        obj2
        }
        frame_name = "Commerce_sell"
        roleset_name = "sell.01 (가정)"
    else:
        # 알 수 없는 동사 유형 → 빈 매핑
        fr_roles = {}
        pb_roles = {}
        frame_name = "Unknown"
        roleset_name = "Unknown"

    # ===== 4) 보기 좋은 출력 =====
    print("\n[예문] ", s)
    print(f"\n[FrameNet 관점: {frame_name}]")
    if fr_roles:
        for k, v in fr_roles.items():
            print(f" - {k:<12}: {v}")
    else:
        print(" - 이 데모 규칙으로는 역할을 붙이기 어렵습니다.")

    print(f"\n[PropBank 관점: {roleset_name}]")
    if pb_roles:
        for k, v in pb_roles.items():
            print(f" - {k:<18}: {v}")
    else:
        print(" - 이 데모 규칙으로는 Arg 매핑이 어렵습니다.")

In [33]:
demo_assign_roles("Mary gave John a book.")
demo_assign_roles("John sent a letter to Mary.")
demo_assign_roles("He sold the car to Mary.")


[예문]  Mary gave John a book.

[FrameNet 관점: Giving / Sending]
 - Giver/Sender: Mary
 - Recipient   : John
 - Goods       : a book

[PropBank 관점: give.01 / send.01 (가정)]
 - Arg0 (Agent)      : Mary
 - Arg1 (Theme)      : a book
 - Arg2 (Recipient)  : John

[예문]  John sent a letter to Mary.

[FrameNet 관점: Giving / Sending]
 - Giver/Sender: John
 - Recipient   : Mary
 - Goods       : a letter

[PropBank 관점: give.01 / send.01 (가정)]
 - Arg0 (Agent)      : John
 - Arg1 (Theme)      : a letter
 - Arg2 (Recipient)  : Mary

[예문]  He sold the car to Mary.

[FrameNet 관점: Commerce_sell]
 - Seller      : He
 - Buyer       : Mary
 - Goods       : the car

[PropBank 관점: sell.01 (가정)]
 - Arg0 (Seller/Agent): He
 - Arg1 (Thing sold) : the car
 - Arg2 (Buyer)      : Mary
